In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import os
from torchvision import transforms
from torch.utils.data import DataLoader
from torchsummary import summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score,roc_auc_score

In [2]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
base_path = "/kaggle/input/datasets/lakshminarayana2409/pneumonia"

print("Base Path Exists:", os.path.exists(base_path))

print(os.listdir(base_path))
print(os.listdir(base_path + "/Training"))
print(os.listdir(base_path + "/Training/Images")[:5])

Base Path Exists: True
['stage2_train_metadata.csv', 'stage2_test_metadata.csv', 'Training', 'Test']
['Images', 'Masks']
['7a530f4d-13af-4e4e-bf2e-080c7fb27ffb.png', '86f084ef-539c-404b-9a91-c9a5d5ab3771.png', '673989cb-131a-43fc-8276-710c65ddb2d6.png', '85a77a2c-9fb8-4ceb-af2b-4de716d4d366.png', '9f756055-99fc-4a80-97b4-f89f98cf92d2.png']


In [4]:
import pandas as pd

csv_path = "/kaggle/input/datasets/lakshminarayana2409/pneumonia/stage2_train_metadata.csv"

print("CSV Exists:", os.path.exists(csv_path))

df = pd.read_csv(csv_path)

df = df[['patientId', 'Target']]

print(df.head())
print(df.shape)

CSV Exists: True
                              patientId  Target
0  0004cfab-14fd-4e49-80ba-63a80b6bddd6       0
1  00313ee0-9eaa-42f4-b0ab-c148ed3241cd       0
2  00322d4d-1c29-4943-afc9-b6754be640eb       0
3  003d8fa0-6bf1-40ed-b54c-ac657f8495c5       0
4  00436515-870c-4b36-a041-de91049b9ab4       1
(30227, 2)


In [5]:
img_dir = base_path + "/Training/Images"

sample_id = df.iloc[0, 0]
img_path = os.path.join(img_dir, sample_id + ".png")

from PIL import Image

image = Image.open(img_path)
print("Original size:", image.size)


Original size: (1024, 1024)


In [6]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(0.5),
    transforms.RandomRotation(degrees=7),
    transforms.RandomAffine(degrees = 0,translate = (0.03,0.03),scale = (0.97,1.03)),
    transforms.ColorJitter( brightness = 0.5,contrast = 0.5 ),
    transforms.GaussianBlur(kernel_size = 3, sigma = (0.1, 1.0) ),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.5],std = [0.5])
])

transform_val = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [7]:
from torch.utils.data import Dataset

class PneumoniaDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]["patientId"]
        label = self.df.iloc[idx]["Target"]

        img_path = os.path.join(self.img_dir, img_id + ".png")

        image = Image.open(img_path).convert("L")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

In [8]:
dataset = PneumoniaDataset(df, img_dir, transform)
print("Dataset size:", len(dataset))

Dataset size: 30227


In [9]:
img, label = dataset[0]

print("Image shape:", img.shape)
print("Label:", label)

Image shape: torch.Size([1, 224, 224])
Label: tensor(0.)


In [10]:
loader = DataLoader(dataset, batch_size = 32, shuffle=True)

images, labels = next(iter(loader))

print("Batch images:", images.shape)
print("Batch labels:", labels.shape)

Batch images: torch.Size([32, 1, 224, 224])
Batch labels: torch.Size([32])


In [11]:
print(df['Target'].value_counts(normalize=True) * 100)

Target
0    68.389188
1    31.610812
Name: proportion, dtype: float64


In [12]:
from torch.utils.data import random_split

# Total dataset size
dataset_size = len(dataset)

# Splits
train_size = int(0.7 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size

# Random split
train_data, val_data, test_data = random_split(
    dataset,
    [train_size, val_size, test_size]
)

# Print sizes
print(f"Training Data Size: {len(train_data)}")
print(f"Validation Data Size: {len(val_data)}")
print(f"Test Data Size: {len(test_data)}")

Training Data Size: 21158
Validation Data Size: 4534
Test Data Size: 4535


## Splitting data into batches

In [13]:
train_load = DataLoader(train_data, batch_size = 32, num_workers = 4,pin_memory = True, shuffle = True)
val_load   = DataLoader(val_data,   batch_size = 32, num_workers = 4,pin_memory = True, shuffle = False)
test_load  = DataLoader(test_data,  batch_size = 32, num_workers = 4,pin_memory = True, shuffle = False)

## SEBlock

In [14]:
class SEBlock(nn.Module):

    def __init__(self, channels, reduction = 16):

        super(SEBlock, self).__init__()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc   = nn.Sequential(

            nn.Linear(
                channels,
                channels // reduction,
                bias=False
            ),

            nn.SiLU(inplace=True),

            nn.Linear(
                channels // reduction,
                channels,
                bias=False
            ),

            nn.Sigmoid()
        )

    def forward(self, x):


        batch_size, channels, _, _ = x.size()

        y = self.pool(x).view(
            batch_size,
            channels
        )


        y = self.fc(y).view(
            batch_size,
            channels,
            1,
            1
        )

        return x * y

In [15]:
class pneumoniaCNN(nn.Module):
    def __init__(self,in_channels = 1):
        super(pneumoniaCNN,self).__init__()

        self.stem = nn.Sequential(
            nn.Conv2d( in_channels , out_channels = 64, kernel_size = 7, padding = 3, stride = 2 ),
            nn.BatchNorm2d( 64 ),
            nn.SiLU( inplace = True ),

            nn.Conv2d ( 64, 64, kernel_size = 3, padding = 1 ),
            nn.BatchNorm2d( 64 ),
            nn.SiLU( inplace = True ),
                
            nn.MaxPool2d( kernel_size = 2, stride = 2)
      )

        # Block 1 
        
        self.block1 = nn.Sequential(
            nn.Conv2d ( in_channels = 64, out_channels = 128, kernel_size = 3, padding = 1 ),
            nn.BatchNorm2d( 128 ),
            nn.SiLU(),

            nn.Conv2d( in_channels = 128, out_channels = 128, kernel_size = 3, padding = 1),
            nn.BatchNorm2d( 128 ),
        )

        self.block1_se   = SEBlock( 128 )
        self.block1_skip = nn.Sequential(
            nn.Conv2d ( 64, 128, kernel_size = 1 ),
            nn.BatchNorm2d ( 128 ),
        )      

        # Block 2

        self.block2 = nn.Sequential(
            nn.Conv2d( in_channels = 128, out_channels = 256, kernel_size = 3, padding = 1 ),
            nn.BatchNorm2d( 256 ),
            nn.SiLU(),
    
            nn.Conv2d( in_channels = 256, out_channels = 256, kernel_size = 3, padding = 1),
            nn.BatchNorm2d( 256 ),
        )

        self.block2_se   = SEBlock( 256 )
        self.block2_skip = nn.Sequential(
            nn.Conv2d( 128, 256, kernel_size = 1 ),
            nn.BatchNorm2d( 256 )
        )
     
        # Block 3

        self.block3 = nn.Sequential(
            nn.Conv2d( in_channels = 256, out_channels = 512, kernel_size = 3, padding = 1 ),
            nn.BatchNorm2d( 512 ),
            nn.SiLU(),
    
            nn.Conv2d( in_channels = 512, out_channels = 512, kernel_size = 3, padding = 1 ),
            nn.BatchNorm2d( 512 )
        )

        self.block3_se   = SEBlock ( 512 )
        self.block3_skip = nn.Sequential(
            nn.Conv2d( 256, 512, kernel_size = 1 ),
            nn.BatchNorm2d( 512 )
        )
        
        # Block 4

        self.block4 = nn.Sequential(
            nn.Conv2d( in_channels = 512, out_channels = 1024, kernel_size = 3, padding = 1),
            nn.BatchNorm2d( 1024 ),
            nn.SiLU(),
    
            nn.Conv2d( in_channels = 1024, out_channels = 1024, kernel_size = 3, padding = 1),
            nn.BatchNorm2d( 1024 ),
        )

        self.block4_se   = SEBlock ( 1024 )
        self.block4_skip = nn.Sequential(
            nn.Conv2d( 512, 1024, kernel_size = 1 ),
            nn.BatchNorm2d( 1024 )
        )
        
        self.gap = nn.AdaptiveAvgPool2d((1,1))
        
       # Classifier (Fully Connected Layers)

        self.classifier = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(1024, 256),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
            
        )
    
    def forward(self, x):

        x = self.stem(x)
        
        # Block 1

        identity = self.block1_skip(x)
        out      = self.block1(x)
        out      = self.block1_se(out)
        x        = F.silu(out + identity)
        
        # Block 2
        
        identity = self.block2_skip(x)
        out      = self.block2(x)
        out      = self.block2_se(out)
        x        = F.silu(out + identity)

        # Block 3

        identity = self.block3_skip(x)
        out      = self.block3(x)
        out      = self.block3_se(out) 
        x        = F.silu(out + identity)

        # Block 4

        identity = self.block4_skip(x) 
        out      = self.block4(x)
        out      = self.block4_se(out)
        x        = F.silu(out + identity)
        
        # Global Average pooling

        x = self.gap(x)
        
        # Flatten Layer
        
        x = torch.flatten(x, start_dim = 1)
        
        # Fully connected Layer 1
        
        x = self.classifier(x)
        
        return x      

## Assigining Architecture to device

In [16]:
model = pneumoniaCNN().to(device)
print(model)

pneumoniaCNN(
  (stem): Sequential(
    (0): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU(inplace=True)
    (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): SiLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block1): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): SiLU()
    (3): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (block1_se): SEBlock(
    (pool): AdaptiveAvgPool2d(output_size=1)
    (fc): Sequential(
      (0): Linear(in_feat

## Model Summary

In [17]:
from torchinfo import summary
summary(model, input_size=(1, 1, 224, 224))

Layer (type:depth-idx)                   Output Shape              Param #
pneumoniaCNN                             [1, 1]                    --
├─Sequential: 1-1                        [1, 64, 56, 56]           --
│    └─Conv2d: 2-1                       [1, 64, 112, 112]         3,200
│    └─BatchNorm2d: 2-2                  [1, 64, 112, 112]         128
│    └─SiLU: 2-3                         [1, 64, 112, 112]         --
│    └─Conv2d: 2-4                       [1, 64, 112, 112]         36,928
│    └─BatchNorm2d: 2-5                  [1, 64, 112, 112]         128
│    └─SiLU: 2-6                         [1, 64, 112, 112]         --
│    └─MaxPool2d: 2-7                    [1, 64, 56, 56]           --
├─Sequential: 1-2                        [1, 128, 56, 56]          --
│    └─Conv2d: 2-8                       [1, 128, 56, 56]          8,320
│    └─BatchNorm2d: 2-9                  [1, 128, 56, 56]          256
├─Sequential: 1-3                        [1, 128, 56, 56]          --
│ 

In [18]:
neg_count = (df['Target'] == 0).sum()
pos_count = (df['Target'] == 1).sum()
computed_pos_weight = neg_count / pos_count

print(f"Negative: {neg_count} | Positive: {pos_count} | pos_weight: {computed_pos_weight:.4f}")

pos_weight = torch.tensor([computed_pos_weight]).to(device)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

Negative: 20672 | Positive: 9555 | pos_weight: 2.1635


In [19]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [20]:
num_epochs = 50

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6
)

 ## Training Epoches

In [21]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

torch.cuda.empty_cache()

In [22]:
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
best_val_acc  = 0.0
best_val_f1   = 0.0
best_epoch    = 0

for epoch in range(num_epochs):

    # ── TRAINING ──────────────────────────────────────────────────────
    model.train()

    running_loss = 0.0
    correct      = 0
    total        = 0

    for images, labels in train_load:
        images = images.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()

        outputs = model(images).squeeze(1)
        loss    = criterion(outputs, labels)
        loss.backward()

        # FIX: Gradient clipping — prevents exploding gradients in deep network
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        running_loss += loss.item()
        preds         = (torch.sigmoid(outputs) > 0.5).float()
        correct      += (preds == labels).sum().item()
        total        += labels.size(0)

    train_loss = running_loss / len(train_load)
    train_acc  = 100.0 * correct / total

    # ── VALIDATION (inside epoch loop — was missing before) ───────────
    model.eval()

    val_loss    = 0.0
    val_correct = 0
    val_total   = 0
    all_preds   = []
    all_labels  = []
    all_probs   = []

    with torch.no_grad():
        for images, labels in val_load:
            images = images.to(device)
            labels = labels.float().to(device)

            outputs = model(images).squeeze(1)
            loss    = criterion(outputs, labels)

            val_loss += loss.item()

            probs  = torch.sigmoid(outputs)
            preds  = (probs > 0.5).float()

            val_correct += (preds == labels).sum().item()
            val_total   += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    avg_val_loss = val_loss / len(val_load)
    val_acc      = 100.0 * val_correct / val_total

    # FIX: F1 + AUC-ROC for reliable evaluation on imbalanced X-ray data
    val_f1  = f1_score(all_labels, all_preds, zero_division=0)
    val_auc = roc_auc_score(all_labels, all_probs)

    # FIX: scheduler.step() now called every epoch (cosine annealing)
    scheduler.step()

    # Save best model checkpoint
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_f1  = val_f1
        best_epoch   = epoch + 1
        torch.save(model.state_dict(), 'best_pneumonia_model.pth')

    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"  Train  — Loss: {train_loss:.4f} | Acc: {train_acc:.2f}%")
    print(f"  Val    — Loss: {avg_val_loss:.4f} | Acc: {val_acc:.2f}% | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
    print(f"  LR: {current_lr:.2e}  |  Best Val Acc: {best_val_acc:.2f}% (Epoch {best_epoch})")
    print("-" * 70)

Epoch [1/50]
  Train  — Loss: 0.8030 | Acc: 68.74%
  Val    — Loss: 0.7362 | Acc: 75.69% | F1: 0.6521 | AUC: 0.8158
  LR: 9.99e-05  |  Best Val Acc: 75.69% (Epoch 1)
----------------------------------------------------------------------
Epoch [2/50]
  Train  — Loss: 0.7122 | Acc: 74.78%
  Val    — Loss: 0.6986 | Acc: 77.42% | F1: 0.6705 | AUC: 0.8382
  LR: 9.96e-05  |  Best Val Acc: 77.42% (Epoch 2)
----------------------------------------------------------------------
Epoch [3/50]
  Train  — Loss: 0.6789 | Acc: 76.55%
  Val    — Loss: 0.7549 | Acc: 79.40% | F1: 0.6757 | AUC: 0.8476
  LR: 9.91e-05  |  Best Val Acc: 79.40% (Epoch 3)
----------------------------------------------------------------------
Epoch [4/50]
  Train  — Loss: 0.6736 | Acc: 76.84%
  Val    — Loss: 0.6806 | Acc: 76.44% | F1: 0.6716 | AUC: 0.8418
  LR: 9.84e-05  |  Best Val Acc: 79.40% (Epoch 3)
----------------------------------------------------------------------
Epoch [5/50]
  Train  — Loss: 0.6618 | Acc: 76.89%
 

## Evaluating Model Performance

In [ ]:
model.load_state_dict(torch.load('best_pneumonia_model.pth'))
model.eval()

test_correct = 0
test_total   = 0
test_loss    = 0.0
test_preds   = []
test_labels  = []
test_probs   = []

with torch.no_grad():
    for images, labels in test_load:
        images = images.to(device)
        labels = labels.float().to(device)

        outputs = model(images).squeeze(1)
        loss    = criterion(outputs, labels)

        test_loss += loss.item()

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        test_correct += (preds == labels).sum().item()
        test_total   += labels.size(0)

        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())
        test_probs.extend(probs.cpu().numpy())

avg_test_loss = test_loss / len(test_load)
test_acc      = 100.0 * test_correct / test_total
test_f1       = f1_score(test_labels, test_preds, zero_division=0)
test_auc      = roc_auc_score(test_labels, test_probs)

print("=" * 50)
print("FINAL TEST SET RESULTS (Best Checkpoint)")
print("=" * 50)
print(f"Test Loss     : {avg_test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.2f}%")
print(f"Test F1 Score : {test_f1:.4f}")
print(f"Test AUC-ROC  : {test_auc:.4f}")
print("=" * 50)